# Part 1 — Exercise solutions

In [ ]:
import json
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../../.env", override=True)
client = genai.Client()
MODEL = "gemini-3.5-flash-lite"

## 7.1 — Persona

In [ ]:
blacksmith = types.GenerateContentConfig(
    system_instruction=(
        "You are Borin, a grumpy medieval blacksmith who somehow reviews modern "
        "video games. You compare everything to honest smithing work, complain "
        "about 'glowing rectangles', and grade games in horseshoes out of 5."
    ),
    temperature=1.0,
)

for game in ["a farming simulator", "a fast arcade racing game"]:
    r = client.models.generate_content(
        model=MODEL, contents=f"Review {game} in 3 sentences.", config=blacksmith
    )
    print(r.text, "\n")

## 7.2 — Temperature bingo

In [ ]:
prompt_stable = "What is 7 times 8? Reply with only the number."
prompt_chaotic = "Invent a single made-up word for the joy of finding a rare item in a game. Reply with only the word."

for temp, prompt in [(0.0, prompt_stable), (2.0, prompt_chaotic)]:
    outs = [
        client.models.generate_content(
            model=MODEL, contents=prompt,
            config=types.GenerateContentConfig(temperature=temp),
        ).text.strip()
        for _ in range(5)
    ]
    print(f"temp {temp}: {outs} → {len(set(outs))} unique")

## 7.3 — JSON by politeness

Expect a decent success rate — but not 100%, and *that's the point*. The failures
(code fences, prose preambles) are exactly what `response_schema` eliminates in Part 4.

In [ ]:
ok = 0
for i in range(5):
    r = client.models.generate_content(
        model=MODEL,
        contents=(
            "Review the imaginary game 'Llama Mechanic 3000'. Reply ONLY with JSON: "
            '{"title": string, "score": number 1-10, "verdict": string}'
        ),
        config=types.GenerateContentConfig(temperature=1.5),  # high temp to stress it
    )
    try:
        json.loads(r.text)
        ok += 1
        print(f"run {i + 1}: ✅")
    except json.JSONDecodeError:
        print(f"run {i + 1}: 💥 → {r.text[:60]!r}…")
print(f"\n{ok}/5 parsed")

## 7.4 — Token economics

In [ ]:
paragraph = (
    "The new update completely changed the balance of the game. Weapons that "
    "used to be strong are now useless, and the community is not happy about it."
)

ro = client.models.generate_content(
    model=MODEL,
    contents=f"Translate to natural Romanian, reply with only the translation:\n{paragraph}",
).text.strip()

en_tokens = client.models.count_tokens(model=MODEL, contents=paragraph).total_tokens
ro_tokens = client.models.count_tokens(model=MODEL, contents=ro).total_tokens

print(ro, "\n")
print(f"EN: {en_tokens} tokens · RO: {ro_tokens} tokens · ratio {ro_tokens / en_tokens:.2f}×")